# 🧪 Sesión 31 · Contenedorización Básica con Docker y Dockerfiles

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Cómo empaquetas tu API para que corra igual en cualquier lado?

---

## 🖥️ Así se verá el resultado final

```text
$ docker build -t biblioteca-api .
[+] Building 12.4s (9/9) FINISHED
 => [1/5] FROM python:3.11-slim
 => [2/5] WORKDIR /app
 => [3/5] COPY requirements.txt .
 => [4/5] RUN pip install --no-cache-dir -r requirements.txt
 => [5/5] COPY . .

$ docker run --rm -p 8000:8000 biblioteca-api
$ curl http://127.0.0.1:8000/api/libros
HTTP/1.1 200 OK
{"libros":[]}

$ docker image inspect biblioteca-api --format '{{.Config.User}}'
appuser
```

_La imagen no debe contener el archivo .env ni ejecutarse como root._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Escribir un Dockerfile reproducible.
- Separar dependencias del código para aprovechar la caché.
- Verificar que la imagen no corre como root ni filtra secretos.

## ✅ Criterios de aceptación

El verificador comprobará estos criterios sobre los archivos que produzcas:

- [ ] Dockerfile reproducible y sin secretos
- [ ] Usuario sin privilegios y arranque con uvicorn
- [ ] Contexto de construcción limpio
- [ ] Dependencias declaradas

## 🧩 El reto

Escribe el Dockerfile y el .dockerignore de la API de libros para que la imagen sea pequeña, reproducible y segura.

**Requisitos del contrato**

- La imagen parte de una versión fija de Python slim.
- Las dependencias se copian e instalan antes de copiar el código.
- El contenedor se ejecuta con un usuario sin privilegios.
- El comando de arranque usa uvicorn y expone el puerto 8000.
- .dockerignore excluye .git, __pycache__ y .env.
- El Dockerfile no copia el archivo .env ni define secretos.

> **Pista conceptual:** El orden importa: `COPY requirements.txt` antes de `COPY . .` permite que Docker reutilice la capa de dependencias cuando solo cambia el código.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion31.md` con el tamaño de la imagen y el usuario de ejecución.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `requirements.txt`, `Dockerfile`, `.dockerignore`


### 📄 `requirements.txt`

Dependencias de la aplicación.

In [2]:

%%writefile requirements.txt
fastapi
uvicorn[standard]
sqlalchemy


Writing requirements.txt


### 📄 `Dockerfile`

Receta de la imagen.

In [3]:
%%writefile Dockerfile
FROM python:3.12-slim

# Evita archivos .pyc y salida de logs con búfer
ENV PYTHONDONTWRITEBYTECODE=1 \
    PYTHONUNBUFFERED=1

WORKDIR /app

# Primero las dependencias: esta capa se reutiliza mientras requirements.txt no cambie
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt

# Usuario sin privilegios
RUN adduser --disabled-password --gecos "" appuser

# Código de la aplicación (respeta .dockerignore)
COPY . .
RUN chown -R appuser:appuser /app

USER appuser

EXPOSE 8000

CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]


Writing Dockerfile


### 📄 `.dockerignore`

Archivos excluidos del contexto de construcción.

In [4]:
%%writefile .dockerignore
.git
.gitignore
__pycache__/
*.pyc
.env
.venv/
venv/
*.db
.ipynb_checkpoints/
Dockerfile
.dockerignore


Writing .dockerignore


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Framework, versión y restricciones del entorno. |
| **Rol** | Qué papel técnico debe asumir el modelo. |
| **Especificación** | Rutas, entradas, salidas, validaciones y códigos HTTP. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [5]:
from verificar import revisar_archivo, resumen

resultados = []
resultados.append(revisar_archivo("Dockerfile", prohibir=["COPY .env", "USER root", "ENV SECRET"], regex=["FROM\\s+python:3\\.1[12]-slim", "WORKDIR\\s+/app", "COPY\\s+requirements\\.txt", "RUN\\s+pip\\s+install", "COPY\\s+\\.\\s+\\."], descripcion="Dockerfile reproducible y sin secretos"))
resultados.append(revisar_archivo("Dockerfile", regex=["USER\\s+appuser", "adduser|useradd", "CMD.*uvicorn|ENTRYPOINT.*uvicorn"], descripcion="Usuario sin privilegios y arranque con uvicorn"))
resultados.append(revisar_archivo(".dockerignore", exigir=[".git", "__pycache__", ".env"], descripcion="Contexto de construcción limpio"))
resultados.append(revisar_archivo("requirements.txt", regex=["fastapi", "uvicorn"], descripcion="Dependencias declaradas"))
resumen(resultados)


✅ Dockerfile reproducible y sin secretos
✅ Usuario sin privilegios y arranque con uvicorn
✅ Contexto de construcción limpio
✅ Dependencias declaradas
------------------------------------------------------------------------------
RESULTADO: 4/4 comprobaciones correctas. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion31.md` a `auditorias/auditoria_sesion31.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
